# Exercise 0: How Does an LLM API Call Work?

Prerequisite: Setup from README completed.

In [1]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b not found: {models}"
print("OK")

OK


## Part 0: From messages to ChatResponse

How `ollama.chat()` hands your message to the LLM and turns its answer back into a structured response.

In [5]:
import requests
OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

# 1. INPUT
messages = [{"role": "user", "content": "What is 2 + 2?"}]
print("1. Input messages:")
print(messages)

# 2. RENDER -- Ollama's Qwen35Renderer turns messages into one prompt string.
rendered = requests.post(f"{OLLAMA_URL}/api/chat", json={
    "model": "qwen3.5:4b", "messages": messages, "_debug_render_only": True,
}).json()["_debug_info"]["rendered_template"]
print("\n2. Prompt Ollama handed to the LLM:")
print(rendered)

# 3. LLM -- raw=True skips the parser; we see the raw tokens the LLM produced.
raw_text = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b", "prompt": rendered, "raw": True, "stream": False,
}).json()["response"]
print("\n3. Raw text the LLM produced:")
print(raw_text)

# 4. PARSE -- Ollama's Qwen35Parser.Add takes the raw text and returns
#    (content, thinking, tool_calls). This is the function doing the mapping:
import urllib.request, re, json as _json
SHA = "3b43b9bc4b1779b5bab0cad4bc077ffb9006a6a1"  # pinned Ollama commit so the fetched qwen35.go line numbers stay stable
src = urllib.request.urlopen(
    f"https://raw.githubusercontent.com/ollama/ollama/{SHA}/model/parsers/qwen35.go"
).read().decode().splitlines()
print("\n4a. Ollama's Qwen35Parser.Add  (model/parsers/qwen35.go:85-107):")
print("\n".join(src[84:107]))

# 4b. The delimiters come straight from the Go source -- we pull the const block
#     so there's no copy-pasting from us. These are the tokens the parser watches for.
print("\n4b. Delimiter constants in qwen35.go (lines 22-24):")
print("\n".join(src[21:24]))

delims = dict(re.findall(r'qwen35(\w+)\s*=\s*"([^"]+)"', "\n".join(src[21:24])))
THINK_OPEN, THINK_CLOSE, TOOL_OPEN = delims["ThinkingOpenTag"], delims["ThinkingCloseTag"], delims["ToolCallOpenTag"]

# 4c. Apply the same logic in Python on step 3's raw_text. Our rendered prompt already
#     contains <think>, so the parser starts in qwen35ParserStateCollectingThinking.
thinking, sep, rest = raw_text.partition(THINK_CLOSE)
content_raw = rest.lstrip() if sep else ""
tool_calls = [_json.loads(m) for m in
              re.findall(rf"{re.escape(TOOL_OPEN)}\s*(\{{.*?\}})\s*</tool_call>", content_raw, re.DOTALL)]
content = re.sub(rf"{re.escape(TOOL_OPEN)}.*?</tool_call>", "", content_raw, flags=re.DOTALL).strip()

print("\n4c. Parser output on step 3's raw_text:")
print(f"     thinking   ({len(thinking):>4} chars): {thinking.strip()[:80]!r}{'...' if len(thinking) > 80 else ''}")
print(f"     content    ({len(content):>4} chars): {content!r}")
print(f"     tool_calls ({len(tool_calls)}):            {tool_calls}")

# 5. STRUCTURED RESPONSE -- ollama.chat() runs RENDER + LLM + PARSE end-to-end and
#    returns the same (content, thinking, tool_calls) fields, now as a ChatResponse.
import json
response = ollama.chat(model="qwen3.5:4b", messages=messages)
print("\n5. ollama.chat() ChatResponse:")
print(json.dumps(response.model_dump(), indent=2, default=str))

1. Input messages:
[{'role': 'user', 'content': 'What is 2 + 2?'}]

2. Prompt Ollama handed to the LLM:
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>


3. Raw text the LLM produced:
Thinking Process:

1.  **Analyze the Request:** The user is asking "What is 2 + 2?". This is a straightforward mathematical addition problem.

2.  **Determine the Answer:** 2 + 2 equals 4.

3.  **Formulate the Output:** Keep it simple and direct. "4" or "The answer is 4."

4.  **Review for Constraints:** No special constraints detected. Just answer the question.

5.  **Final Output Generation:** 2 + 2 = 4.cw
</think>

2 + 2 = 4

4a. Ollama's Qwen35Parser.Add  (model/parsers/qwen35.go:85-107):
func (p *Qwen35Parser) Add(s string, done bool) (content string, thinking string, calls []api.ToolCall, err error) {
	p.buffer.WriteString(s)
	events := p.parseEvents()

	var contentSb strings.Builder
	var thinkingSb strings.Builder
	for _, event := range events {
		switch event := event.(type

## Part 1: A Single LLM Call

An LLM generates text, token by token. Everything we see in APIs like `ollama.chat()` (fields like `thinking`, `content`, `tool_calls`) is parsed from this text stream after the fact.

First, let's look at the raw output directly via the REST API.

In [ ]:
# Raw LLM output directly via the REST API
# raw=true: no parsing by Ollama, we see the text as it is generated
# Prompt in the Qwen3.5 chat template with <think> pre-fill (activates think mode)
import requests, json

OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWhat is 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# <think> was pre-filled, so prepend it for the complete picture
raw_output = "<think>\n" + resp.json()["response"]
# print(repr(raw_output))
print(raw_output)

In [ ]:
# Without chat template: the model doesn't know where the answer should stop
# We abort after 3 seconds
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "What is 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

# print(repr("".join(tokens)))
print("".join(tokens))

In [ ]:
# ollama.chat() does the same thing, but:
# 1. Builds the prompt with special tokens automatically (RENDERER)
# 2. Parses <think>...</think> from the output into the "thinking" field (PARSER)
# 3. The rest becomes "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "What is 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below. Run it. Briefly note what changed and why.

1. **Remove** the `<think>\n` from the prompt in cell 3. What disappears from the output?
2. **Change** the prompt in cell 4 to `"Answer once, then stop: What is 2 + 2?"`. Does it still run away? What does this tell you about what actually stops generation?
3. **Add** `think=False` to the `ollama.chat(...)` call in cell 5. Which field in the response is now empty, and which one grew?


In [ ]:
# Your Turn — Part 1
# Copy the relevant cell above, paste it here, modify, and run.


## Part 2: Statelessness

The chat API is stateless. The model only sees the `messages` we send in each call.

In [ ]:
# Without conversation history: two separate calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "What is my name?"}])
print("Call 2:", r2.message.content)

In [ ]:
# With conversation history: same two calls, but we pass the history along
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "My name is Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "My name is Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "What is my name?"},
])
print("Call 2:", r2.message.content)

---
### Your Turn

Make each change in the scratchpad below. Run it. Briefly note what changed and why.

1. **Remove** the `{"role": "assistant", ...}` entry from the message list in cell 8's second call. Does the model still know the name? Why?
2. **Replace** `r1.message.content` in the history with the hard-coded string `"Nice to meet you, Anna!"`. What does the model think your name is?
3. **Move** the fact `"My name is Max."` out of the first `user` message and into a `system` message at the start of the list. Send only `"What is my name?"` as the user turn (no assistant history). Does it still work? Which role do instructions and stable facts conventionally live in, and why is that different from facts the user states mid-conversation?


In [ ]:
# Your Turn — Part 2
# Copy the relevant cell above, paste it here, modify, and run.


## Part 3: Tool Calling

An LLM always only generates text. Tool calling is not a special feature, but a **convention for structured output**: the model was trained to output JSON instead of free text for certain prompts.

We can first build this by hand, using only a system prompt.

In [ ]:
# "Tool calling" without the tools= parameter, using only a prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'When you need to calculate, respond ONLY with this JSON:\n{"tool": "calculator", "expression": "<expression>"}\nNo other text.'},
        {"role": "user", "content": "Calculate 17 * 23"},
    ],
)

print("=== Raw Output (just text) ===")
print(response.message.content)

In [ ]:
# We can parse the text output ourselves and execute the tool
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# This is tool calling: the model outputs structured text, we parse and execute.
print(parsed["expression"])
result = eval(parsed["expression"])
print("Result:", result)

This works, but it's fragile: we have to formulate the prompt precisely, define the JSON format ourselves, and parse the output manually. The `tools=` parameter automates exactly this:
- It injects the tool definitions into the prompt (in the format the model saw during training)
- It parses the output and delivers structured `tool_calls` instead of raw text

From here on we use `tools=`. The tool definition describes to the model which functions are available. The tool implementation is the Python function we execute.

In [ ]:
# Tool definition: describes to the model which function is available
# Reference: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

In [ ]:
# Call the LLM with the tool definition, inspect the full response
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Calculate 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

In the response we see:
- `content` is empty: the model did not return any text
- `tool_calls` is populated: the model wants to call `calculator` with `{"expression": "17 * 23"}`

The model did not calculate anything. It returned **which function** it wants to call with **which arguments**. Now we need the Python function that actually executes it.

In [ ]:
# Tool implementation: the Python function we execute
import numexpr, math

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: call directly
print(calculator("17 * 23"))

In [ ]:
# Connection: how do we find the right function for a tool call?
# The name in the tool call ("calculator") must match the key in tool_map.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool call from the response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # Look up the calculator function
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

---
### Your Turn

Make each change in the scratchpad below. Run it. Briefly note what changed and why.

1. **Change** the `description` in `calculator_tool` (cell 13) to an empty string `""`, then rerun cell 14 with prompt `"Calculate 17 * 23"`. Does the model still call the tool?
2. **Change** the user prompt in cell 14 to `"What is 2 + 2?"`. Does the model call the tool for trivial math, or answer directly? What's in `content` vs `tool_calls`?
3. **Rename** the tool in cell 13 from `"calculator"` to `"math_thing"` but leave `tool_map = {"calculator": calculator}` unchanged in cell 17. Rerun cell 17. What error do you get, and at which line?


In [ ]:
# Your Turn — Part 3
# Copy the relevant cell above, paste it here, modify, and run.


## Part 4: Tool-Use Loop

We have executed the tool and got a result. But the API is stateless: the model knows nothing about the execution. We must include the result as a `{"role": "tool"}` message in the history and call the model again.

![The tool-use loop animated](loop_animation.gif)

*Watch the `messages` list grow by one row per step. The loop keeps running while the LLM's response contains `tool_calls`. That cycle is what the next code cell implements.*

In [ ]:
# Complete tool-use loop
messages = [{"role": "user", "content": "Calculate 17 * 23"}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

In [ ]:
# Sanity check: what happens when the model decides AGAINST using a tool?
messages = [{"role": "user", "content": "Calculate 17 * 23. Answer directly, do NOT use a tool."}]

# 1. Call the LLM
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Check: does the model want to call a tool?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Execute the tool
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    # ** unpacks the dict into keyword arguments:
    # {"expression": "17 * 23"} becomes calculator(expression="17 * 23")
    result = func(**tc.function.arguments)

    # 4. Send the tool result back to the model
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages to LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Final Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

---
### Your Turn

Make each change in the scratchpad below. Run it. Briefly note what changed and why.

1. **Remove** the line `messages.append(response.message.model_dump())` from cell 19 (keep the tool result append). Rerun. What does the final response look like and why?
2. **Change** the prompt in cell 19 to `"Calculate 17 * 23, then divide the result by 2."` and wrap the tool-call block in `while response.message.tool_calls:` with a safety cap of 5 iterations. How many tool calls does the model make — one combined expression, or two sequential calls?
3. **Change** the prompt in cell 20 to `"Tell me a short joke."` (keep `tools=calculator_tool` unchanged). Does the model touch the calculator? Inspect `content` and `tool_calls` — what does the model do when the available tools are irrelevant to the task, and what does that tell you about how tool selection is decided?


In [ ]:
# Your Turn — Part 4
# Copy the relevant cell above, paste it here, modify, and run.


## Summary

1. `ollama.chat()` takes a `messages` list and returns a `ChatResponse` object.
2. The API is stateless. The model only sees the messages we send in each call.
3. With `tools=`, the model can return tool calls. We check for this with `if response.message.tool_calls`.
4. The tool definition (JSON object with name, description, parameter schema) tells the model what is available. The tool implementation (Python function) is executed by us. They are connected via `tool_map`.
5. The tool-use loop (call LLM → check → execute tool → send result back → call LLM again) is what an agent automates.

→ **Exercise 1**: implement this loop.